In [1]:
# Load all the environment variables from ~/.bashrc

import os

paths = []
if os.path.join(os.path.expanduser('~'), '.bashrc'):
    paths.append(os.path.join(os.path.expanduser('~'), '.bashrc'))
if os.path.join(os.path.expanduser('~'), '.zshrc'):
    paths.append(os.path.join(os.path.expanduser('~'), '.zshrc'))

for path in paths:
    with open(path) as f:
        lines = f.readlines()
        for line in lines:
            if line.startswith('export'):
                var = line.split()[1].split('=')[0]
                if var == "PATH":
                    continue
                os.environ[var] = line.split("=")[1].strip().strip('"')
                print(var, os.environ[var])

OPENSSL_ROOT_DIR /usr/local/opt/openssl@3
ZSH /Users/guillemcv/.oh-my-zsh
PL_API_KEY PLAKfda103372c864062b8900aaa4c9639bd
LUPNT_DATA_PATH /Users/guillemcv/Development/NavLab/LuPNT/data
CLOUDSDK_PYTHON $(which python)
CODECOV_TOKEN d0f7e7f7-f53c-4a81-8d30-2539383403f5
GMAT_PATH /Users/guillemcv/Applications/GMAT R2022a


In [2]:
import pylupnt as pnt
import numpy as np
import os
from utils.gmat import gmat, gmat_path
from utils import data

kwargs = {}
coe = data.coe_array_elfo
earth_potential_file_path = os.path.join(
    gmat_path, "data", "gravity", "earth", "JGM3.cof"
)
luna_potential_file_path = os.path.join(
    gmat_path, "data", "gravity", "luna", "grgm900c.cof"
)

EarthMJ2000Eq = gmat.Construct("CoordinateSystem", "EarthMJ2000Eq", "Earth", "MJ2000Eq")
LunaMJ2000Eq = gmat.Construct("CoordinateSystem", "LunaMJ2000Eq", "Luna", "MJ2000Eq")

# Spacecraft configuration preliminaries
sc = gmat.Construct("Spacecraft", "LunaOrbiter")
sc.SetField("DateFormat", "UTCGregorian")
sc.SetField("Epoch", "20 Jul 2020 12:00:00.000")
sc.SetField("CoordinateSystem", "LunaMJ2000Eq")
# sc.SetField("DisplayStateType", "Keplerian")

# Orbital state
sc.SetField("SMA", coe[0])
sc.SetField("ECC", coe[1])
sc.SetField("INC", np.rad2deg(coe[2]))
sc.SetField("RAAN", np.rad2deg(coe[3]))
sc.SetField("AOP", np.rad2deg(coe[4]))
sc.SetField("TA", np.rad2deg(pnt.mean_to_true_anomaly(coe[5], coe[1])))

# Spacecraft ballistic properties for the SRP and Drag models
if "SRPArea" in kwargs:
    sc.SetField("SRPArea", 2.5)
if "Cr" in kwargs:
    sc.SetField("Cr", 1.75)
if "DragArea" in kwargs:
    sc.SetField("DragArea", 1.8)
if "Cd" in kwargs:
    sc.SetField("Cd", 2.1)

sc.SetField("DryMass", 80)

# Force model settings
fm = gmat.Construct("ForceModel", "FM")
fm.SetField("CentralBody", "Luna")

# An 8x8 JGM-3 Gravity Model
grav = gmat.Construct("GravityField")
grav.SetField("BodyName", "Luna")
grav.SetField("PotentialFile", luna_potential_file_path)
grav.SetField("Degree", 0)
grav.SetField("Order", 0)

# Add forces into the ODEModel container
fm.AddForce(grav)

gmat.Initialize()

# Build the propagation container class
pdprop = gmat.Construct("Propagator", "PDProp")

# Create and assign a numerical integrator for use in the propagation
gator = gmat.Construct("PrinceDormand78", "Gator")
pdprop.SetReference(gator)

# Set some of the fields for the integration
pdprop.SetField("InitialStepSize", 60.0)
pdprop.SetField("Accuracy", 1.0e-12)
pdprop.SetField("MinStep", 0.0)

# Perform top level initialization
gmat.Initialize()

# Setup the spacecraft that is propagated
pdprop.AddPropObject(sc)
pdprop.PrepareInternals()

# Refresh the 'gator reference
fm = pdprop.GetODEModel()
gator = pdprop.GetPropagator()

# Take a 60 second step, showing the state before and after propagation
print(sc.GetEpoch())
print(gator.GetState())
print(sc.GetState().GetState())
print(sc.GetCartesianState())

29051.000428638676
[-162550.00497932723, 305071.7865004737, 155929.03244103043, -1.0433590712049405, -1.3773987417408444, 0.12840375330478881]
[-162550.00497932723, 305071.7865004737, 155929.03244103043, -1.0433590712049405, -1.3773987417408444, 0.12840375330478881]
-2857.618683217181 -2997.994311239047 5731.496383057383 -0.1222975129350246 -0.8992864018240367 0.2452908630394531


In [3]:
dyn = pnt.CartesianTwoBodyDynamics(pnt.MU_MOON)

cart = pnt.classical_to_cartesian(utils.coe_array_elfo, pnt.MU_MOON)

print(cart)
(cart2, stm) = dyn.propagate(cart, 0, 30 * 60, 1)
print(cart2)

[-2.85761868e+03 -2.99799431e+03  5.73149638e+03 -1.22297506e-01
 -8.99286354e-01  2.45290850e-01]


ValueError: too many values to unpack (expected 2)

In [4]:
print(sc.GetCartesianState())

-2857.618683217181 -2997.994311239047 5731.496383057383 -0.1222975129350246 -0.8992864018240367 0.2452908630394531


In [5]:
epoch = gmat.GmatTime(sc.GetEpoch())
state_earth = gmat.Rvector6(*gator.GetState())
state_luna = gmat.Rvector6()

In [6]:
print(type(epoch))
print(type(state_earth))
print(type(state_luna))
print(type(EarthMJ2000Eq))
print(type(LunaMJ2000Eq))

<class '_py39.gmat_py.GmatTime'>
<class '_py39.gmat_py.Rvector6'>
<class '_py39.gmat_py.Rvector6'>
<class 'gmat_py.CoordinateSystem'>
<class 'gmat_py.CoordinateSystem'>


In [7]:
converter = gmat.CoordinateConverter()
converter.Convert(epoch, state_earth, EarthMJ2000Eq, state_luna, LunaMJ2000Eq)
print(state_luna)

-2857.618683105946 -2997.994311181305 5731.496383071499 -0.1222975129351668 -0.8992864018237618 0.2452908630395873
